# 13 — Inspect the Raw Iceberg Files (metadata, manifest list, manifests, parquet)

An Iceberg table on S3 is a tree of files. The chain from a query down to bytes is:

```
metadata.json                          ← table state: schema, partition spec, list of snapshots
   └─ snapshot record (inside .json)
        └─ manifest_list (snap-*.avro) ← list of manifest files in this snapshot
              └─ manifest (*.avro)     ← list of data files in this manifest's scope
                    └─ data file (*.parquet)   ← actual rows
```

On Nessie, **every commit writes a new `metadata.json`**, but the Iceberg-level snapshot
list inside it has only one entry — the rest of the version history lives in Nessie's
commit log (see notebook 12).

This notebook reads each layer directly from S3 and shows you exactly what's inside.


In [1]:
import json
import os
from collections import Counter

import boto3
import polars as pl
import pyarrow.parquet as pq
import requests
from pyiceberg.avro.file import AvroFile
from pyiceberg.catalog.rest import RestCatalog
from pyiceberg.io.pyarrow import PyArrowFileIO

NESSIE_URI       = os.environ["NESSIE_URI"]
NESSIE_API       = NESSIE_URI.replace("/iceberg", "/api/v2")
S3_ENDPOINT      = os.environ["AWS_S3_ENDPOINT"]
S3_ACCESS_KEY    = os.environ["AWS_ACCESS_KEY_ID"]
S3_SECRET_KEY    = os.environ["AWS_SECRET_ACCESS_KEY"]
WAREHOUSE_BUCKET = os.environ["ICEBERG_WAREHOUSE_BUCKET"]

S3_PROPS = {
    "uri": NESSIE_URI,
    "warehouse": f"s3://{WAREHOUSE_BUCKET}/warehouse",
    "s3.endpoint": S3_ENDPOINT,
    "s3.access-key-id": S3_ACCESS_KEY,
    "s3.secret-access-key": S3_SECRET_KEY,
    "s3.path-style-access": "true",
    "s3.region": "us-east-1",
}

# Catalog (for high-level table lookups) and FileIO (for reading individual files)
catalog = RestCatalog(name="nessie", **S3_PROPS)
fileio  = PyArrowFileIO(properties=S3_PROPS)

# Plain boto3 client for listing and reading raw bytes
s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=S3_ACCESS_KEY,
    aws_secret_access_key=S3_SECRET_KEY,
    region_name="us-east-1",
)


def s3_split(uri: str) -> tuple[str, str]:
    """s3://bucket/key/path → (bucket, key/path)"""
    assert uri.startswith("s3://")
    bucket, _, key = uri[5:].partition("/")
    return bucket, key


def read_avro(s3_uri: str, limit: int | None = None) -> list:
    """Read an Avro file via PyIceberg's reader; return up to `limit` records."""
    with AvroFile(fileio.new_input(s3_uri)) as af:
        out = []
        for i, rec in enumerate(af):
            if limit is not None and i >= limit: break
            out.append(rec)
        return out


print("Ready.")

Ready.


## 1. Pick a target table

Change `TARGET` to inspect any table in your catalog. `bronze.weather_raw` is the
default because it has many commits (one per daily ETL run).

In [2]:
TARGET = ("bronze", "weather_raw")

table = catalog.load_table(TARGET)
print("table:           ", ".".join(TARGET))
print("table location:  ", table.location())
print("metadata pointer:", table.metadata_location)
print("current snapshot:", table.current_snapshot().snapshot_id)

table:            bronze.weather_raw
table location:   s3://iceberg-warehouse/warehouse/bronze/weather_raw_ae2180aa-1a78-4947-9647-f5ec8108796c
metadata pointer: s3://iceberg-warehouse/warehouse/bronze/weather_raw_ae2180aa-1a78-4947-9647-f5ec8108796c/metadata/00000-6b86d43f-3d90-41d3-bd3c-f9ea90c4903d.metadata.json
current snapshot: 7351079934310758371


## 2. List every `metadata.json` ever written for this table

Each Nessie commit produces a new `*.metadata.json` (different UUID per commit).
They all live under `<table_location>/metadata/` and are *never deleted* until you
run `nessie-gc`.

In [3]:
bucket, table_key = s3_split(table.location())
metadata_prefix = f"{table_key}/metadata/"

resp = s3.list_objects_v2(Bucket=bucket, Prefix=metadata_prefix)
all_files = resp.get("Contents", [])

metadata_jsons = sorted(
    [o for o in all_files if o["Key"].endswith(".metadata.json")],
    key=lambda o: o["LastModified"],
)
manifest_lists = [o for o in all_files if "/snap-" in o["Key"] and o["Key"].endswith(".avro")]
manifests      = [o for o in all_files if o["Key"].endswith(".avro") and "/snap-" not in o["Key"]]

print(f"  metadata.json files : {len(metadata_jsons)}")
print(f"  manifest list (snap-*.avro) : {len(manifest_lists)}")
print(f"  manifest files (*-m*.avro)  : {len(manifests)}")
print()
print("Latest 5 metadata.json files:")
for o in metadata_jsons[-5:]:
    print(f"  {o['LastModified']}  {o['Size']:>6} B   {o['Key'].split('/')[-1]}")

  metadata.json files : 12
  manifest list (snap-*.avro) : 11
  manifest files (*-m*.avro)  : 11

Latest 5 metadata.json files:
  2026-05-23 10:57:37.194000+00:00    2299 B   00000-1ea476b9-5181-4c26-980f-bd9c3e0f09f0.metadata.json
  2026-05-23 11:01:35.157000+00:00    2300 B   00000-fc24a684-d2f9-46d3-a1cc-33c4678709f0.metadata.json
  2026-05-23 11:02:15.836000+00:00    2300 B   00000-f7282b7e-8aee-45a9-a2fa-5a86eb52fc4a.metadata.json
  2026-05-23 11:02:26.538000+00:00    2303 B   00000-335377f0-cce2-45c7-8aa9-375e4bf3f8f3.metadata.json
  2026-05-23 11:02:42.532000+00:00    2303 B   00000-6b86d43f-3d90-41d3-bd3c-f9ea90c4903d.metadata.json


## 3. Read one `metadata.json` (raw JSON)

The top-level table state. Important keys: `format-version`, `location`,
`current-snapshot-id`, `snapshots` (list — on Nessie it has exactly one),
`schemas`, `partition-specs`, `properties`.

In [4]:
# Read the CURRENT metadata (the one Nessie points at). To inspect a past
# version, change `metadata_key` to one of the older entries above.
metadata_key = s3_split(table.metadata_location)[1]

obj = s3.get_object(Bucket=bucket, Key=metadata_key)
metadata = json.loads(obj["Body"].read())

print("format-version    :", metadata["format-version"])
print("table-uuid        :", metadata["table-uuid"])
print("location          :", metadata["location"])
print("last-sequence-num :", metadata["last-sequence-number"])
print("current-snapshot  :", metadata["current-snapshot-id"])
print("schemas           :", len(metadata["schemas"]),     "(latest has", len(metadata["schemas"][-1]["fields"]), "fields)")
print("partition-specs   :", len(metadata["partition-specs"]))
print("snapshots         :", len(metadata["snapshots"]),   "  ← Nessie collapses this to 1")
print("properties keys   :", list(metadata.get("properties", {}).keys())[:5])

format-version    : 2
table-uuid        : dcaa7e3b-03a5-407c-bc32-58bc34713b01
location          : s3://iceberg-warehouse/warehouse/bronze/weather_raw_ae2180aa-1a78-4947-9647-f5ec8108796c
last-sequence-num : 11
current-snapshot  : 7351079934310758371
schemas           : 1 (latest has 13 fields)
partition-specs   : 1
snapshots         : 1   ← Nessie collapses this to 1
properties keys   : ['gc.enabled', 'created-at']


## 4. The snapshot record (lives inside `metadata.json`)

Each snapshot row carries a `manifest-list` path, an operation summary (added/removed
files and rows), and a `timestamp-ms`. This is the bridge to the next layer.

In [5]:
snap = metadata["snapshots"][-1]
print("snapshot-id      :", snap["snapshot-id"])
print("parent-snapshot  :", snap.get("parent-snapshot-id"))
print("sequence-number  :", snap.get("sequence-number"))
print("timestamp-ms     :", snap["timestamp-ms"])
print("manifest-list    :", snap["manifest-list"])
print("summary          :")
for k, v in snap["summary"].items():
    print(f"    {k}: {v}")

snapshot-id      : 7351079934310758371
parent-snapshot  : None
sequence-number  : 11
timestamp-ms     : 1779534162440
manifest-list    : s3://iceberg-warehouse/warehouse/bronze/weather_raw_ae2180aa-1a78-4947-9647-f5ec8108796c/metadata/snap-7351079934310758371-0-abfadbd9-cb63-45ad-b9af-49a245417942.avro
summary          :
    operation: append
    added-files-size: 12508
    added-data-files: 1
    added-records: 24
    changed-partition-count: 1
    total-data-files: 11
    total-delete-files: 0
    total-records: 264
    total-files-size: 137712
    total-position-deletes: 0
    total-equality-deletes: 0


## 5. Read the manifest list (`snap-*.avro`)

This Avro file lists every manifest that belongs to this snapshot. Each row points
to a `.avro` manifest plus partition summary and counts.

In [6]:
manifest_list_uri = snap["manifest-list"]
mlist = read_avro(manifest_list_uri)
print(f"{len(mlist)} manifest entries in {manifest_list_uri.split('/')[-1]}\n")

rows = []
for m in mlist:
    rows.append({
        "manifest_path": m.manifest_path.split("/")[-1],
        "length_bytes": m.manifest_length,
        "added_snapshot_id": m.added_snapshot_id,
        "added_data_files_count": getattr(m, "added_data_files_count", None) or getattr(m, "added_files_count", None),
        "existing_data_files_count": getattr(m, "existing_data_files_count", None) or getattr(m, "existing_files_count", None),
        "sequence_number": getattr(m, "sequence_number", None),
    })

pl.DataFrame(rows)

11 manifest entries in snap-7351079934310758371-0-abfadbd9-cb63-45ad-b9af-49a245417942.avro



manifest_path,length_bytes,added_snapshot_id,added_data_files_count,existing_data_files_count,sequence_number
str,i64,i64,i64,i64,i64
"""abfadbd9-cb63-45ad-b9af-49a245…",5565,7351079934310758371,1,0,11
"""d1f58737-f731-4a4e-8ae4-cd6234…",5564,2549601228501353997,1,0,10
"""81216801-bf07-451e-8185-20bfa1…",5564,1868881171936492771,1,0,9
"""fa71d717-84b3-4e72-9334-fc0270…",5565,7596267586117503414,1,0,8
"""b5f54a8f-877f-47e3-8163-dc2146…",5564,2950741286297060510,1,0,7
…,…,…,…,…,…
"""a8fe1d26-3b91-459b-b2a0-e50eed…",5565,7567649627025845988,1,0,5
"""b9198a19-13c8-4fb7-9f5c-936f91…",5565,6727912641296334357,1,0,4
"""85de8f65-c6ce-4132-ad16-dd083f…",5565,5741000150922061192,1,0,3


## 6. Read one manifest file (`*-m*.avro`)

Each manifest enumerates data files (parquet) in its scope, with per-file stats:
record count, byte size, partition value, and column lower/upper bounds. Iceberg uses
these bounds for predicate-pushdown without touching parquet.

In [7]:
# Pick the first manifest from the manifest list
first_manifest_uri = mlist[0].manifest_path
entries = read_avro(first_manifest_uri)
print(f"{len(entries)} entries in {first_manifest_uri.split('/')[-1]}\n")

# Show file-level info for each entry's data_file
df_rows = []
for e in entries:
    d = e.data_file
    df_rows.append({
        "status": {0: "EXISTING", 1: "ADDED", 2: "DELETED"}.get(e.status, e.status),
        "snapshot_id": e.snapshot_id,
        "file_path": d.file_path.split("/")[-1],
        "format": str(d.file_format),
        "records": d.record_count,
        "size_bytes": d.file_size_in_bytes,
        "partition": str(d.partition),
    })

pl.DataFrame(df_rows)

1 entries in abfadbd9-cb63-45ad-b9af-49a245417942-m0.avro



status,snapshot_id,file_path,format,records,size_bytes,partition
str,i64,str,str,i64,i64,str
"""ADDED""",7351079934310758371,"""00000-0-abfadbd9-cb63-45ad-b9a…","""PARQUET""",24,12508,"""Record[ingest_day=20596]"""


## 7. Per-column stats inside one manifest entry

`lower_bounds` and `upper_bounds` are per-column min/max as raw bytes (Iceberg stores
them encoded per Iceberg type). These power partition-aware filter pushdown.

In [8]:
d = entries[0].data_file
schema_fields = {f.field_id: f.name for f in table.schema().fields}

stats_rows = []
for fid, name in schema_fields.items():
    stats_rows.append({
        "field_id": fid,
        "name": name,
        "null_count": (d.null_value_counts or {}).get(fid),
        "value_count": (d.value_counts or {}).get(fid),
        "column_size": (d.column_sizes or {}).get(fid),
        "lower_bytes": (d.lower_bounds or {}).get(fid),
        "upper_bytes": (d.upper_bounds or {}).get(fid),
    })

pl.DataFrame(stats_rows).head(10)

field_id,name,null_count,value_count,column_size,lower_bytes,upper_bytes
i64,str,i64,i64,i64,binary,binary
1,"""city""",0,24,76,"b""Baku""","b""Baku"""
2,"""lat""",0,24,110,"b""\\x20A\xf1c4D@""","b""\\x20A\xf1c4D@"""
3,"""lon""",0,24,110,"b""I.\xff!\xfd\xeeH@""","b""I.\xff!\xfd\xeeH@"""
4,"""observation_ts""",0,24,269,"b""\x00\x80\x97\xd3pR\x06\x00""","b""\x00<\xdb\x1a\x84R\x06\x00"""
5,"""temperature_2m""",0,24,211,"b""\xcd\xcc\xcc\xcc\xcc\xcc/@""","b""\x00\x00\x00\x00\x00\x807@"""
6,"""relative_humidity_2m""",0,24,186,"b""\x00\x00\x00\x00\x00\x00K@""","b""\x00\x00\x00\x00\x00@W@"""
7,"""wind_speed_10m""",0,24,211,"b""333333\x0b@""","b""\x00\x00\x00\x00\x00\x00,@"""
8,"""precipitation""",0,24,110,"b""\x00\x00\x00\x00\x00\x00\x00\x80""","b""\x00\x00\x00\x00\x00\x00\x00\x00"""
9,"""pressure_msl""",0,24,202,"b""\xcd\xcc\xcc\xcc\xcc\xb4\x8f@""","b""\x00\x00\x00\x00\x00\xcc\x8f@"""


## 8. Read the parquet data file

PyArrow reads parquet directly from S3 via the same FileIO. We dump the parquet
schema plus a sample of rows.

In [9]:
data_file_uri = entries[0].data_file.file_path
inp = fileio.new_input(data_file_uri)

# pyarrow.parquet expects a NativeFile / Python file-like; pull bytes from S3 directly.
bucket_d, key_d = s3_split(data_file_uri)
buf = s3.get_object(Bucket=bucket_d, Key=key_d)["Body"].read()
print(f"parquet file: {data_file_uri.split('/')[-1]}  ({len(buf):,} bytes)\n")

import io
pf = pq.ParquetFile(io.BytesIO(buf))
print("Schema:")
print(pf.schema_arrow)
print()
print(f"num_rows: {pf.metadata.num_rows}")
print(f"num_row_groups: {pf.metadata.num_row_groups}")
print(f"created_by: {pf.metadata.created_by}")
print()
print("Sample:")
pl.from_arrow(pf.read_row_group(0)).head(5)

parquet file: 00000-0-abfadbd9-cb63-45ad-b9af-49a245417942.parquet  (12,508 bytes)

Schema:
city: large_string not null
  -- field metadata --
  PARQUET:field_id: '1'
lat: double
  -- field metadata --
  PARQUET:field_id: '2'
lon: double
  -- field metadata --
  PARQUET:field_id: '3'
observation_ts: timestamp[us, tz=UTC] not null
  -- field metadata --
  PARQUET:field_id: '4'
temperature_2m: double
  -- field metadata --
  PARQUET:field_id: '5'
relative_humidity_2m: double
  -- field metadata --
  PARQUET:field_id: '6'
wind_speed_10m: double
  -- field metadata --
  PARQUET:field_id: '7'
precipitation: double
  -- field metadata --
  PARQUET:field_id: '8'
pressure_msl: double
  -- field metadata --
  PARQUET:field_id: '9'
cloud_cover: double
  -- field metadata --
  PARQUET:field_id: '10'
ingest_ts: timestamp[us, tz=UTC] not null
  -- field metadata --
  PARQUET:field_id: '11'
source: large_string not null
  -- field metadata --
  PARQUET:field_id: '12'
raw_payload: large_string
  -- f

city,lat,lon,observation_ts,temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation,pressure_msl,cloud_cover,ingest_ts,source,raw_payload
str,f64,f64,"datetime[μs, UTC]",f64,f64,f64,f64,f64,f64,"datetime[μs, UTC]",str,str
"""Baku""",40.4093,49.8671,2026-05-23 00:00:00 UTC,15.9,91.0,7.6,0.0,1016.3,89.0,2026-05-23 11:02:36.810267 UTC,"""open-meteo""","""{""latitude"":40.4375,""longitude…"
"""Baku""",40.4093,49.8671,2026-05-23 01:00:00 UTC,15.9,93.0,7.8,0.0,1016.4,99.0,2026-05-23 11:02:36.810267 UTC,"""open-meteo""","""{""latitude"":40.4375,""longitude…"
"""Baku""",40.4093,49.8671,2026-05-23 02:00:00 UTC,15.9,93.0,5.7,0.0,1016.9,96.0,2026-05-23 11:02:36.810267 UTC,"""open-meteo""","""{""latitude"":40.4375,""longitude…"
"""Baku""",40.4093,49.8671,2026-05-23 03:00:00 UTC,16.8,90.0,8.0,0.0,1016.9,81.0,2026-05-23 11:02:36.810267 UTC,"""open-meteo""","""{""latitude"":40.4375,""longitude…"
"""Baku""",40.4093,49.8671,2026-05-23 04:00:00 UTC,17.8,85.0,6.4,0.0,1017.5,82.0,2026-05-23 11:02:36.810267 UTC,"""open-meteo""","""{""latitude"":40.4375,""longitude…"


## 9. Walk every Nessie version of this table

This stitches notebooks 12 and 13: for each Nessie commit that touched the table,
fetch the table at that hash, then count the files it points to. The result is a
per-version footprint table.

In [10]:
def iter_main_commits(page_size=100, max_pages=50):
    params = {"maxRecords": page_size}
    for _ in range(max_pages):
        body = requests.get(f"{NESSIE_API}/trees/main/history", params=params).json()
        for entry in body["logEntries"]:
            yield entry["commitMeta"]
        token = body.get("token")
        if not token:
            return
        params = {"maxRecords": page_size, "pageToken": token}


needle = f"ICEBERG_TABLE {TARGET[0]}.{TARGET[1]}"
table_commits = [c for c in iter_main_commits() if needle in c.get("message", "")]
print(f"{len(table_commits)} commits for {TARGET[0]}.{TARGET[1]}\n")

rows = []
for i, cm in enumerate(table_commits):
    h = cm["hash"]
    try:
        cat_h = RestCatalog(name=f"v{i}", **{**S3_PROPS, "uri": f"{NESSIE_URI}/main@{h}"})
        t_h = cat_h.load_table(TARGET)
        snap_h = t_h.current_snapshot()
        ml = read_avro(snap_h.manifest_list) if snap_h else []
        added = sum((getattr(m, "added_data_files_count", None) or
                     getattr(m, "added_files_count", 0)) or 0 for m in ml)
        existing = sum((getattr(m, "existing_data_files_count", None) or
                        getattr(m, "existing_files_count", 0)) or 0 for m in ml)
        rows.append({
            "i": i,
            "hash": h[:12],
            "commit_time": cm["commitTime"],
            "metadata_file": t_h.metadata_location.split("/")[-1],
            "snapshot_id": snap_h.snapshot_id if snap_h else None,
            "manifests": len(ml),
            "data_files_added": added,
            "data_files_existing": existing,
        })
    except Exception as e:
        rows.append({"i": i, "hash": h[:12], "commit_time": cm["commitTime"],
                     "metadata_file": f"<error: {type(e).__name__}>",
                     "snapshot_id": None, "manifests": None,
                     "data_files_added": None, "data_files_existing": None})

pl.DataFrame(rows)

12 commits for bronze.weather_raw



i,hash,commit_time,metadata_file,snapshot_id,manifests,data_files_added,data_files_existing
i64,str,str,str,i64,i64,i64,i64
0,"""1ec4a339b7a9""","""2026-05-23T11:02:42.539062757Z""","""00000-6b86d43f-3d90-41d3-bd3c-…",7351079934310758371,11,11,0
1,"""00ea9884c8f3""","""2026-05-23T11:02:26.542752180Z""","""00000-335377f0-cce2-45c7-8aa9-…",2549601228501353997,10,10,0
2,"""5461b5cc73e3""","""2026-05-23T11:02:15.845271300Z""","""00000-f7282b7e-8aee-45a9-a2fa-…",1868881171936492771,9,9,0
3,"""91a58dbd824c""","""2026-05-23T11:01:35.163001128Z""","""00000-fc24a684-d2f9-46d3-a1cc-…",7596267586117503414,8,8,0
4,"""fdecd50367a6""","""2026-05-23T10:57:37.198118671Z""","""00000-1ea476b9-5181-4c26-980f-…",2950741286297060510,7,7,0
…,…,…,…,…,…,…,…
7,"""d323db7b5007""","""2026-05-23T10:56:32.049029669Z""","""00000-4924048e-cddf-47f3-91c3-…",6727912641296334357,4,4,0
8,"""fb3cead54a1e""","""2026-05-23T10:55:31.059524126Z""","""00000-608deb30-e50c-429d-a667-…",5741000150922061192,3,3,0
9,"""dcd222dd32ef""","""2026-05-23T10:55:21.954739553Z""","""00000-ed84a157-2301-4e26-8ac1-…",4891444363665842195,2,2,0


## 10. (Optional) Estimate storage footprint of one version

For a chosen Nessie commit, walk manifests → data files and sum the bytes. Useful
when deciding whether a particular version is worth keeping vs. running `nessie-gc`.

In [11]:
def footprint_at_commit(commit_hash: str) -> dict:
    cat_h = RestCatalog(name="fp", **{**S3_PROPS, "uri": f"{NESSIE_URI}/main@{commit_hash}"})
    t_h = cat_h.load_table(TARGET)
    snap_h = t_h.current_snapshot()
    if snap_h is None:
        return {"hash": commit_hash[:12], "data_files": 0, "records": 0, "bytes": 0}

    total_files, total_records, total_bytes = 0, 0, 0
    for m in read_avro(snap_h.manifest_list):
        for entry in read_avro(m.manifest_path):
            total_files   += 1
            total_records += entry.data_file.record_count
            total_bytes   += entry.data_file.file_size_in_bytes
    return {"hash": commit_hash[:12], "data_files": total_files,
            "records": total_records, "bytes": total_bytes}


if table_commits:
    fp = footprint_at_commit(table_commits[0]["hash"])
    print(f"Footprint @ {fp['hash']} (latest):")
    print(f"  data files : {fp['data_files']}")
    print(f"  records    : {fp['records']:,}")
    print(f"  bytes      : {fp['bytes']:,}  ({fp['bytes']/1024:.1f} KiB)")

Footprint @ 1ec4a339b7a9 (latest):
  data files : 11
  records    : 264
  bytes      : 137,712  (134.5 KiB)
